# Geographic-Scale Dataset Build

Builds a place-based dataset for the scaling sweep:

- The top-N (default 10) species at the place are the **target pool**. Each gets its own class folder, downloaded with a large per-species budget.
- All other species present in the area are pooled into `non_target`, with a small per-species download budget *and* a per-species cap when assembling the pool so no single non-top species dominates.
- AudioSet ambient + BirdNET no-bird clips are added to `non_target` for negative diversity.

Top-N species are **never** placed in `non_target` — they only ever appear as targets in `train.ipynb`. All downloads are cached: re-running only fetches what's missing.

In [12]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
from pathlib import Path

import pyrootutils

ROOT = pyrootutils.setup_root(
    search_from=Path.cwd(),
    indicator="pyproject.toml",
    pythonpath=True,
    dotenv=True,
)

# Parameters

In [14]:
from building.geographic_scale import place_slug

PLACE_NAME = "Paris"
N_TARGETS = 10

# Area discovery coordinates — recorded in dataset.json so downstream
# notebooks never need to redefine them.
LAT, LON = 48.8566, 2.3522
RADIUS_KM = 50

# Asymmetric per-species download budget: top-N get the full budget, the
# rest get a small fixed quota each. download_and_process is idempotent
# and budget-bounded, so it never re-fetches a species already cached at
# >= the requested quota.
TARGET_CLIPS_PER_SPECIES = 5000
NON_TARGET_CLIPS_PER_SPECIES = 250

# Hard cap on each non-top species' contribution to the on-disk non_target
# class. Defends against the case where a non-top species was downloaded
# heavily in another collection and now has many more clips on disk than
# the per-species download quota above.
NON_TARGET_PER_SPECIES_CAP = 300

# Area discovery (matches analysis.ipynb defaults)
MIN_AREA_RECORDINGS = 10
ANALYSIS_MAX_BIRDNET = 2000
ANALYSIS_MIN_CONF = 0.85

# AudioSet ambient/anthropic noise (diversified across ~60 FOCUS_CLASSES)
AS_CLIPS_PER_CLASS = 250
AS_MAX_TOTAL_CLIPS = 15000

# Slug = "<place>_<n_targets>" (e.g. "paris_10"). One name — call it from
# every other notebook.
COLLECTION_NAME = place_slug(PLACE_NAME, N_TARGETS)
print(f"collection: {COLLECTION_NAME}")

collection: paris_10


# Discover area species and pick top-N as targets

`combined_species_table` returns the area's species sorted by evidence
(XC presence + BirdNET detections). The top-N become target classes; the
rest are pooled into `non_target`.

In [15]:
from building.analysis import (
    area_audio_cache_dir,
    bbox_from_radius,
    birdnet_area_analysis,
    combined_species_table,
    fetch_area_recordings,
    recordings_to_df,
)

bbox = bbox_from_radius(LAT, LON, RADIUS_KM)
area_df = recordings_to_df(await fetch_area_recordings(bbox))
bn = await birdnet_area_analysis(
    area_df,
    max_recordings=ANALYSIS_MAX_BIRDNET,
    min_confidence=ANALYSIS_MIN_CONF,
    cache_dir=area_audio_cache_dir(LAT, LON, RADIUS_KM),
)
combined = combined_species_table(
    area_df, bn["detections"], min_recordings=MIN_AREA_RECORDINGS
)
all_species = combined["scientific_name"].tolist()
TARGET_SPECIES = all_species[:N_TARGETS]
OTHER_SPECIES = all_species[N_TARGETS:]
print(f"Top-{N_TARGETS} target species:")
for sp in TARGET_SPECIES:
    print(f"  {sp}")
print(f"\n{len(OTHER_SPECIES)} other species pooled into non_target")

BirdNET analyse:   0%|          | 0/786 [00:00<?, ?it/s]

Top-10 target species:
  Anthus spinoletta
  Parus major
  Psittacula krameri
  Hippolais polyglotta
  Dendrocoptes medius
  Fringilla coelebs
  Certhia brachydactyla
  Curruca communis
  Troglodytes troglodytes
  Erithacus rubecula

31 other species pooled into non_target


# Write `dataset.json` (pre-download manifest)

Everything downstream (analysis, training, results) reads this small
file instead of re-deriving species lists from BirdNET + XC. Written
*before* any download so the dataset's identity is captured even if a
download is interrupted.

In [16]:
from building.geographic_scale import DatasetInfo, write_dataset_info
from building.geographic_scale.dataset_info import Audio, Budgets, Place

info = DatasetInfo(
    collection=COLLECTION_NAME,
    place=Place(name=PLACE_NAME, lat=LAT, lon=LON, radius_km=RADIUS_KM),
    n_targets=N_TARGETS,
    target_species=TARGET_SPECIES,
    other_species=OTHER_SPECIES,
    audio=Audio(),  # 16 kHz, 3 s — matches building/utils/constants.py
    budgets=Budgets(
        target_clips_per_species=TARGET_CLIPS_PER_SPECIES,
        non_target_clips_per_species=NON_TARGET_CLIPS_PER_SPECIES,
        non_target_per_species_cap=NON_TARGET_PER_SPECIES_CAP,
        audioset_clips_per_class=AS_CLIPS_PER_CLASS,
        audioset_max_total=AS_MAX_TOTAL_CLIPS,
    ),
)
print(write_dataset_info(info))

/home/nathan/Documents/multi-chirp/datasets/paris_10/dataset.json


# Stream-download target species (heavy budget)

In [17]:
from building.data.download import download_and_process
from building.data.sources import XCSource, EBirdSource

with XCSource() as xc, EBirdSource() as eb:
    for sp in TARGET_SPECIES:
        await download_and_process(sp, TARGET_CLIPS_PER_SPECIES, [xc, eb])


=== [Anthus spinoletta] target=10000 clips (5000/source × 2 sources)  on disk=2375 [xc=1310/5000, ebird=1065/5000] ===
[Anthus spinoletta] --- Phase A: per-source quota (parallel) ---
[Anthus spinoletta / xc] no pending recordings (available=311, already processed=311, on disk=1310/5000).
[Anthus spinoletta / ebird] no pending recordings (available=843, already processed=843, on disk=1065/5000).
[Anthus spinoletta] --- Phase B: top up (have 2375/10000) ---
[Anthus spinoletta / xc] no pending recordings (available=311, already processed=311, on disk=1310/8935).
[Anthus spinoletta / ebird] no pending recordings (available=843, already processed=843, on disk=1065/8690).
[Anthus spinoletta] done. final 2375/10000 clips on disk [xc=1310, ebird=1065].

=== [Parus major] target=10000 clips (5000/source × 2 sources)  on disk=10000 [xc=5000/5000, ebird=5000/5000] ===
[Parus major] already at target, nothing to do.

=== [Psittacula krameri] target=10000 clips (5000/source × 2 sources)  on disk=

[Certhia brachydactyla/ebird] recs:   0%|          | 0/1 [00:00<?, ?it/s]

[Certhia brachydactyla/ebird] clips:  81%|########  | 5429/6724 [00:00<?, ?it/s]

[Certhia brachydactyla / ebird] download failed ML621338550.mp3: 404 Client Error: Not Found for url: https://cdn.download.ams.birds.cornell.edu/api/v1/asset/621338550/audio
[Certhia brachydactyla] done. final 8705/10000 clips on disk [xc=3276, ebird=5429].

=== [Curruca communis] target=10000 clips (5000/source × 2 sources)  on disk=10000 [xc=5000/5000, ebird=5000/5000] ===
[Curruca communis] already at target, nothing to do.

=== [Troglodytes troglodytes] target=10000 clips (5000/source × 2 sources)  on disk=10000 [xc=5000/5000, ebird=5000/5000] ===
[Troglodytes troglodytes] already at target, nothing to do.

=== [Erithacus rubecula] target=10000 clips (5000/source × 2 sources)  on disk=10000 [xc=5000/5000, ebird=5000/5000] ===
[Erithacus rubecula] already at target, nothing to do.


# Stream-download other species (light budget, pooled as non-target)

In [18]:
with XCSource() as xc, EBirdSource() as eb:
    for sp in OTHER_SPECIES:
        await download_and_process(sp, NON_TARGET_CLIPS_PER_SPECIES, [xc, eb])


=== [Regulus ignicapilla] target=500 clips (250/source × 2 sources)  on disk=9897 [xc=3962/250, ebird=5935/250] ===
[Regulus ignicapilla] already at target, nothing to do.

=== [Sylvia atricapilla] target=500 clips (250/source × 2 sources)  on disk=600 [xc=300/250, ebird=300/250] ===
[Sylvia atricapilla] already at target, nothing to do.

=== [Phylloscopus collybita] target=500 clips (250/source × 2 sources)  on disk=600 [xc=300/250, ebird=300/250] ===
[Phylloscopus collybita] already at target, nothing to do.

=== [Locustella luscinioides] target=500 clips (250/source × 2 sources)  on disk=600 [xc=300/250, ebird=300/250] ===
[Locustella luscinioides] already at target, nothing to do.

=== [Acrocephalus palustris] target=500 clips (250/source × 2 sources)  on disk=600 [xc=300/250, ebird=300/250] ===
[Acrocephalus palustris] already at target, nothing to do.

=== [Poecile palustris] target=500 clips (250/source × 2 sources)  on disk=600 [xc=300/250, ebird=300/250] ===
[Poecile palustri

[Corvus corone/ebird] recs:   0%|          | 0/34 [00:00<?, ?it/s]

[Corvus corone/ebird] clips:  86%|########6 | 216/250 [00:00<?, ?it/s]

[Corvus corone] --- Phase B: top up (have 298/500) ---
[Corvus corone / xc] no pending recordings (available=107, already processed=107, on disk=76/278).
[Corvus corone / ebird] 222/424 on disk; queued 202 of 1727 available.


[Corvus corone/ebird] recs:   0%|          | 0/202 [00:00<?, ?it/s]

[Corvus corone/ebird] clips:  52%|#####2    | 222/424 [00:00<?, ?it/s]

[Corvus corone] done. final 313/500 clips on disk [xc=76, ebird=237].

=== [Acanthis cabaret] target=500 clips (250/source × 2 sources)  on disk=500 [xc=500/250, ebird=0/250] ===
[Acanthis cabaret] already at target, nothing to do.

=== [Turdus philomelos] target=500 clips (250/source × 2 sources)  on disk=600 [xc=334/250, ebird=266/250] ===
[Turdus philomelos] already at target, nothing to do.

=== [Luscinia megarhynchos] target=500 clips (250/source × 2 sources)  on disk=600 [xc=300/250, ebird=300/250] ===
[Luscinia megarhynchos] already at target, nothing to do.

=== [Pica pica] target=500 clips (250/source × 2 sources)  on disk=600 [xc=300/250, ebird=300/250] ===
[Pica pica] already at target, nothing to do.

=== [Picus viridis] target=500 clips (250/source × 2 sources)  on disk=600 [xc=300/250, ebird=300/250] ===
[Picus viridis] already at target, nothing to do.

=== [Lophophanes cristatus] target=500 clips (250/source × 2 sources)  on disk=500 [xc=250/250, ebird=250/250] ===
[Lop

# Stream-download AudioSet ambient clips

In [21]:
from building.data.audioset import AudioSetConfig, stream_download_audioset_async

as_cfg = AudioSetConfig(
    clips_per_class=AS_CLIPS_PER_CLASS,
    max_total_clips=AS_MAX_TOTAL_CLIPS,
)
results = await stream_download_audioset_async(as_cfg)

[audioset] split=unbalanced_train per_class=250 global=15000 on_disk=14506 workers=6
[audioset] indexing unbalanced_train segments...
[audioset] candidates per class: Speech=936732, Male speech, man speaking=6941, Female speech, woman speaking=5057, Conversation=2140, Narration, monologue=15253, ...
[Speech] 250 clips, skipping.
[Male speech, man speaking] 250 clips, skipping.
[Female speech, woman speaking] 250 clips, skipping.
[Conversation] 250 clips, skipping.
[Narration, monologue] 250 clips, skipping.
[Children playing] no pending recordings.
[Crowd] 250 clips, skipping.
[Laughter] 250 clips, skipping.
[Applause] 250 clips, skipping.
[Cheering] 250 clips, skipping.
[Whistling] 250 clips, skipping.
[Singing] 250 clips, skipping.
[Walk, footsteps] 250 clips, skipping.
[Door] 250 clips, skipping.
[Knock] no pending recordings.
[Typing] 250 clips, skipping.
[Computer keyboard] 250 clips, skipping.
[Aircraft] 250 clips, skipping.
[Helicopter] 250 clips, skipping.
[Car] 250 clips, skip

# Assemble train/val/test

Each top-N species becomes its own class folder; `non_target` is the
balanced pool of (audioset + xc_other + birdnet_no_bird), with each
non-top species capped at `NON_TARGET_PER_SPECIES_CAP` clips before
pooling. Top-N species are never passed as non-target species, so they
cannot leak into `non_target`.

In [20]:
from building.data.dataset import build_task_dataset

dataset_path = build_task_dataset(
    COLLECTION_NAME,
    target_species=TARGET_SPECIES,
    non_target_species=OTHER_SPECIES,
    non_target_per_species_cap=NON_TARGET_PER_SPECIES_CAP,
)
print(dataset_path)

Non-target pool: 19400 clips (8700 audioset + 8700 xc_other + 2000 birdnet_no_bird) (per-species cap=300)
  audioset:         take 8700 of 14506 available
  xc_other:         take 8700 of 8700 available
  birdnet_no_bird:  take 2000 of 2000 available
Processing Anthus_spinoletta with 2375 clips
Training: 1646, Validation: 382, Testing: 347
Copied 1646 clips from Anthus_spinoletta to training
Copied 382 clips from Anthus_spinoletta to validation
Copied 347 clips from Anthus_spinoletta to testing
Processing Parus_major with 10000 clips
Training: 7000, Validation: 1500, Testing: 1500
Copied 7000 clips from Parus_major to training
Copied 1500 clips from Parus_major to validation
Copied 1500 clips from Parus_major to testing
Processing Psittacula_krameri with 5636 clips
Training: 3945, Validation: 844, Testing: 847
Copied 3945 clips from Psittacula_krameri to training
Copied 844 clips from Psittacula_krameri to validation
Copied 847 clips from Psittacula_krameri to testing
Processing Hippol